# EA-TTM — Validation Ablation and Mechanism Audit

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 09 — Validation ablations, mechanism evidence, and model-development lock  
**Scope:** Explain *why* the trained EA-TTM behaves as observed, quantify the contribution of decoder cross-feed and gating, and freeze all development decisions before the single locked final-test evaluation.

This phase does **not** change the primary Phase 08 model and does **not** evaluate the locked final-test interval.

## Primary model remains fixed

Phase 08 pre-registered the **best balanced checkpoint** as the primary EA-TTM checkpoint. That decision is not revisited in this phase.

The primary validation score was:

$$
S_{\mathrm{balanced}}
=
\frac{1}{2}
\left(
\frac{\mathrm{MSE}_{EA,\mathrm{all}}}
     {\mathrm{MSE}_{HP,\mathrm{all}}}
+
\frac{\mathrm{MSE}_{EA,\mathrm{ext}}}
     {\mathrm{MSE}_{HP,\mathrm{ext}}}
\right).
$$

Phase 09 is diagnostic only.

## Mechanism questions

The audit addresses four questions.

### 1. Does the gate separate extreme from normal observations?

The primary evidence is ranking quality:

- AUROC;
- AUPRC;
- AUPRC lift over the positive-class prevalence.

Because the gate probability is used continuously in

$$
g_h\Delta_h,
$$

a fixed classification threshold such as 0.5 is only a secondary diagnostic.

### 2. Does EA-TTM apply larger *effective* corrections to extremes?

The correction actually applied to the frozen base forecast is:

$$
C_{t,h}
=
g_{t,h}\Delta_{t,h}.
$$

Therefore the relevant mechanism statistic is:

$$
|C_{t,h}|,
$$

not $|\Delta_{t,h}|$ alone.

### 3. Does decoder cross-feed add value?

A **retail-only gated adapter** is trained with the same Phase 08 optimization contract but excludes the 128-dimensional decoder stream:

$$
[\mathbf z_t;\mathbf c_h]
\in
\mathbb{R}^{13}.
$$

This directly tests the incremental contribution of the Slim Decoder cross-feed.

### 4. Does the Extreme Gate add value?

An **ungated cross-feed residual adapter** retains the full 141-dimensional input but uses:

$$
\hat y_h
=
\hat y_{\mathrm{HP},h}
+
\Delta_h.
$$

It is trained only on the normalized forecasting objective. This tests the incremental effect of soft routing plus explicit extreme supervision.

## Scientific constraint

Ablation results are **not** used to replace, retune, or reselect the Phase 08 primary model.

After this notebook, the primary model-development state is frozen:

$$
\boxed{
\text{Primary model}
=
\text{Phase 08 best-balanced EA-TTM}
}
$$

The next phase performs one locked evaluation on:

$$
d_{1914}:d_{1941}.
$$

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch import nn

## Project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"

PHASE05_CACHE_DIR = (
    PROJECT_ROOT
    / "runs"
    / "05_decoder_crossfeed"
    / "cache"
)

PHASE06_FEATURE_DIR = (
    PROJECT_ROOT
    / "runs"
    / "06_retail_features"
    / "features"
)

PHASE07_TARGET_DIR = (
    PROJECT_ROOT
    / "runs"
    / "07_ea_ttm_architecture"
    / "targets"
)

PHASE08_RUN_DIR = (
    PROJECT_ROOT
    / "runs"
    / "08_ea_ttm_training"
)

PHASE08_CHECKPOINT_DIR = (
    PHASE08_RUN_DIR
    / "checkpoints"
)

PHASE08_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "ea_ttm_adapter"
)

RUN_DIR = (
    PROJECT_ROOT
    / "runs"
    / "09_validation_ablation"
)

CHECKPOINT_DIR = RUN_DIR / "checkpoints"

ABLATION_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "ablation_adapters"
)

for directory in (
    RESULTS_TABLE_DIR,
    RUN_DIR,
    CHECKPOINT_DIR,
    ABLATION_MODEL_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_artifacts = {
    "phase04_summary": METADATA_DIR / "04_RUN_SUMMARY.json",
    "phase05_summary": METADATA_DIR / "05_RUN_SUMMARY.json",
    "phase05_cache_manifest": METADATA_DIR / "05_CACHE_MANIFEST.json",
    "decoder_alignment_weights": METADATA_DIR / "05_DECODER_ALIGNMENT_WEIGHTS.npz",
    "phase06_summary": METADATA_DIR / "06_RUN_SUMMARY.json",
    "phase07_summary": METADATA_DIR / "07_RUN_SUMMARY.json",
    "phase08_summary": METADATA_DIR / "08_RUN_SUMMARY.json",
    "phase08_training_contract": METADATA_DIR / "08_TRAINING_CONTRACT.json",
    "phase08_primary_adapter": PHASE08_MODEL_DIR / "adapter_state.pt",
    "phase08_primary_config": PHASE08_MODEL_DIR / "adapter_config.json",
    "phase08_validation_outputs": PHASE08_RUN_DIR / "08_VALIDATION_OUTPUTS.npz",
    "phase08_best_overall": PHASE08_CHECKPOINT_DIR / "best_overall_state.pt",
    "train_decoder_hidden": PHASE05_CACHE_DIR / "train_decoder_hidden.npy",
    "train_base_forecast": PHASE05_CACHE_DIR / "train_base_forecast.npy",
    "validation_decoder_hidden": PHASE05_CACHE_DIR / "validation_decoder_hidden.npy",
    "validation_base_forecast": PHASE05_CACHE_DIR / "validation_base_forecast.npy",
    "train_recent_features": PHASE06_FEATURE_DIR / "train_recent_demand_features.npy",
    "validation_recent_features": PHASE06_FEATURE_DIR / "validation_recent_demand_features.npy",
    "train_future_covariates": PHASE06_FEATURE_DIR / "train_future_covariates.npy",
    "validation_future_covariates": PHASE06_FEATURE_DIR / "validation_future_covariates.npy",
    "train_extreme_labels": PHASE07_TARGET_DIR / "train_extreme_labels.npy",
    "validation_extreme_labels": PHASE07_TARGET_DIR / "validation_extreme_labels.npy",
    "train_residual_targets": PHASE07_TARGET_DIR / "train_residual_targets.npy",
    "validation_residual_targets": PHASE07_TARGET_DIR / "validation_residual_targets.npy",
}

preflight = {
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in required_artifacts.items()
}

missing_artifacts = [
    item["path"]
    for item in preflight.values()
    if not item["exists"]
]

assert not missing_artifacts, (
    "Phase 09 prerequisites are incomplete: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive


## Recover locked contracts

In [3]:
phase04_summary = json.loads(
    (METADATA_DIR / "04_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

phase05_summary = json.loads(
    (METADATA_DIR / "05_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

cache_manifest = json.loads(
    (METADATA_DIR / "05_CACHE_MANIFEST.json").read_text(
        encoding="utf-8"
    )
)

phase06_summary = json.loads(
    (METADATA_DIR / "06_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

phase07_summary = json.loads(
    (METADATA_DIR / "07_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

phase08_summary = json.loads(
    (METADATA_DIR / "08_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

phase08_training_contract = json.loads(
    (METADATA_DIR / "08_TRAINING_CONTRACT.json").read_text(
        encoding="utf-8"
    )
)

phase08_adapter_config = json.loads(
    (PHASE08_MODEL_DIR / "adapter_config.json").read_text(
        encoding="utf-8"
    )
)

DATASET_SHA256 = phase08_summary["dataset_sha256"]
SPLIT_SHA256 = phase08_summary["split_sha256"]
TRAINING_CONFIG_SHA256 = phase08_summary["training_config_sha256"]

TRAINING_SAMPLE_COUNT = int(
    cache_manifest["train_decoder_hidden"]["shape"][0]
)

VALIDATION_SAMPLE_COUNT = int(
    cache_manifest["validation_decoder_hidden"]["shape"][0]
)

FORECAST_HORIZON = 28

assert DATASET_SHA256 == phase07_summary["dataset_sha256"]
assert SPLIT_SHA256 == phase07_summary["split_sha256"]
assert TRAINING_SAMPLE_COUNT == 121_960
assert VALIDATION_SAMPLE_COUNT == 30_490
assert phase08_summary["locked_final_test_used"] is False
assert phase08_summary["primary_selected_epoch"] == phase08_summary["best_balanced_epoch"]
assert phase08_adapter_config["selected_checkpoint"] == "best_balanced"

PRIMARY_EPOCH = int(
    phase08_summary["primary_selected_epoch"]
)

BEST_OVERALL_EPOCH = int(
    phase08_summary["best_overall_epoch"]
)

BEST_EXTREME_EPOCH = int(
    phase08_summary["best_extreme_epoch"]
)

print("Primary balanced epoch:", PRIMARY_EPOCH)
print("Best overall epoch:", BEST_OVERALL_EPOCH)
print("Best extreme epoch:", BEST_EXTREME_EPOCH)

Primary balanced epoch: 15
Best overall epoch: 20
Best extreme epoch: 15


## Reproducibility state

In [4]:
BASE_SEED = 2026

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_global_seed(BASE_SEED)

assert torch.cuda.is_available(), (
    "CUDA GPU is required for Phase 09."
)

DEVICE = torch.device("cuda")

## Load cached development arrays

In [5]:
train_decoder_hidden = np.load(
    PHASE05_CACHE_DIR / "train_decoder_hidden.npy",
    mmap_mode="r",
)

train_base_forecast = np.load(
    PHASE05_CACHE_DIR / "train_base_forecast.npy",
    mmap_mode="r",
)

validation_decoder_hidden = np.load(
    PHASE05_CACHE_DIR / "validation_decoder_hidden.npy",
    mmap_mode="r",
)

validation_base_forecast = np.load(
    PHASE05_CACHE_DIR / "validation_base_forecast.npy",
    mmap_mode="r",
)

train_recent_features = np.load(
    PHASE06_FEATURE_DIR / "train_recent_demand_features.npy",
    mmap_mode="r",
)

validation_recent_features = np.load(
    PHASE06_FEATURE_DIR / "validation_recent_demand_features.npy",
    mmap_mode="r",
)

train_future_covariates = np.load(
    PHASE06_FEATURE_DIR / "train_future_covariates.npy",
    mmap_mode="r",
)

validation_future_covariates = np.load(
    PHASE06_FEATURE_DIR / "validation_future_covariates.npy",
    mmap_mode="r",
)

train_extreme_labels = np.load(
    PHASE07_TARGET_DIR / "train_extreme_labels.npy",
    mmap_mode="r",
)

validation_extreme_labels = np.load(
    PHASE07_TARGET_DIR / "validation_extreme_labels.npy",
    mmap_mode="r",
)

train_residual_targets = np.load(
    PHASE07_TARGET_DIR / "train_residual_targets.npy",
    mmap_mode="r",
)

validation_residual_targets = np.load(
    PHASE07_TARGET_DIR / "validation_residual_targets.npy",
    mmap_mode="r",
)

alignment = np.load(
    METADATA_DIR / "05_DECODER_ALIGNMENT_WEIGHTS.npz"
)

selected_weight = alignment["selected_weight"].astype(
    np.float32,
    copy=False,
)

training_targets = (
    np.asarray(
        train_base_forecast,
        dtype=np.float32,
    )
    + np.asarray(
        train_residual_targets,
        dtype=np.float32,
    )
).astype(
    np.float32
)

validation_targets = (
    np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    )
    + np.asarray(
        validation_residual_targets,
        dtype=np.float32,
    )
).astype(
    np.float32
)

validation_extreme_mask = (
    np.asarray(
        validation_extreme_labels
    ) == 1
)

validation_normal_mask = (
    ~validation_extreme_mask
)

assert selected_weight.shape == (
    28,
    9,
    128,
)

## Metrics

In [6]:
def forecasting_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    actual_64 = actual.astype(
        np.float64,
        copy=False,
    )

    predicted_64 = predicted.astype(
        np.float64,
        copy=False,
    )

    error = (
        predicted_64
        - actual_64
    )

    absolute_error = np.abs(
        error
    )

    squared_error = np.square(
        error
    )

    mse = float(
        squared_error.mean()
    )

    rmse = float(
        np.sqrt(mse)
    )

    mae = float(
        absolute_error.mean()
    )

    denominator = float(
        np.abs(
            actual_64
        ).sum()
    )

    wape = float(
        absolute_error.sum()
        / denominator
    )

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "wape": wape,
    }

def masked_mse(
    actual: np.ndarray,
    predicted: np.ndarray,
    mask: np.ndarray,
) -> float:
    errors = (
        predicted.astype(
            np.float64,
            copy=False,
        )
        - actual.astype(
            np.float64,
            copy=False,
        )
    )

    return float(
        np.square(
            errors[mask]
        ).mean()
    )

def improvement_percent(
    baseline: float,
    candidate: float,
) -> float:
    return float(
        100.0
        * (
            baseline
            - candidate
        )
        / baseline
    )

## Primary Phase 08 mechanism evidence

In [7]:
primary_outputs = np.load(
    PHASE08_RUN_DIR / "08_VALIDATION_OUTPUTS.npz"
)

primary_actual = primary_outputs["actual"].astype(
    np.float32,
    copy=False,
)

primary_hp = primary_outputs["hp_ttm_forecast"].astype(
    np.float32,
    copy=False,
)

primary_ea = primary_outputs["ea_ttm_forecast"].astype(
    np.float32,
    copy=False,
)

primary_gate = primary_outputs["gate_probability"].astype(
    np.float32,
    copy=False,
)

primary_delta = primary_outputs["residual_correction"].astype(
    np.float32,
    copy=False,
)

primary_extreme_label = primary_outputs["extreme_label"].astype(
    np.uint8,
    copy=False,
)

assert np.allclose(
    primary_actual,
    validation_targets,
    rtol=0.0,
    atol=1e-6,
)

assert np.array_equal(
    primary_extreme_label,
    np.asarray(
        validation_extreme_labels,
        dtype=np.uint8,
    ),
)

primary_effective_correction = (
    primary_gate
    * primary_delta
)

primary_extreme_prevalence = float(
    primary_extreme_label.mean(
        dtype=np.float64
    )
)

primary_gate_auprc = float(
    average_precision_score(
        primary_extreme_label.ravel(),
        primary_gate.ravel(),
    )
)

primary_gate_auroc = float(
    roc_auc_score(
        primary_extreme_label.ravel(),
        primary_gate.ravel(),
    )
)

primary_auprc_lift = float(
    primary_gate_auprc
    / primary_extreme_prevalence
)

mean_abs_effective_correction_extreme = float(
    np.abs(
        primary_effective_correction[
            validation_extreme_mask
        ]
    ).mean(
        dtype=np.float64
    )
)

mean_abs_effective_correction_normal = float(
    np.abs(
        primary_effective_correction[
            validation_normal_mask
        ]
    ).mean(
        dtype=np.float64
    )
)

effective_correction_ratio_extreme_to_normal = float(
    mean_abs_effective_correction_extreme
    / mean_abs_effective_correction_normal
)

base_squared_error = np.square(
    primary_hp.astype(
        np.float64
    )
    - primary_actual.astype(
        np.float64
    )
)

ea_squared_error = np.square(
    primary_ea.astype(
        np.float64
    )
    - primary_actual.astype(
        np.float64
    )
)

extreme_point_improvement_rate = float(
    (
        ea_squared_error[
            validation_extreme_mask
        ]
        < base_squared_error[
            validation_extreme_mask
        ]
    ).mean()
)

normal_point_improvement_rate = float(
    (
        ea_squared_error[
            validation_normal_mask
        ]
        < base_squared_error[
            validation_normal_mask
        ]
    ).mean()
)

true_residual = (
    primary_actual
    - primary_hp
)

nonzero_residual_mask = (
    np.abs(
        true_residual
    )
    > 1e-12
)

correction_direction_agreement = float(
    (
        np.sign(
            primary_effective_correction[
                nonzero_residual_mask
            ]
        )
        == np.sign(
            true_residual[
                nonzero_residual_mask
            ]
        )
    ).mean()
)

extreme_nonzero_mask = (
    validation_extreme_mask
    & nonzero_residual_mask
)

extreme_correction_direction_agreement = float(
    (
        np.sign(
            primary_effective_correction[
                extreme_nonzero_mask
            ]
        )
        == np.sign(
            true_residual[
                extreme_nonzero_mask
            ]
        )
    ).mean()
)

print("Gate AUPRC lift over prevalence:", primary_auprc_lift)
print(
    "Effective correction |extreme| / |normal|:",
    effective_correction_ratio_extreme_to_normal,
)

Gate AUPRC lift over prevalence: 3.560126164092149
Effective correction |extreme| / |normal|: 2.185766031235958


## Adapter implementations for checkpoint and ablation evaluation

In [8]:
class GatedAdapter(nn.Module):
    def __init__(
        self,
        input_dim: int,
        extreme_prior: float,
        encoder_width: int = 16,
        correction_hidden_width: int = 8,
    ) -> None:
        super().__init__()

        self.input_dim = int(
            input_dim
        )

        self.shock_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                encoder_width,
            ),
            nn.GELU(),
            nn.LayerNorm(
                encoder_width,
            ),
        )

        self.extreme_gate = nn.Linear(
            encoder_width,
            1,
        )

        self.correction_head = nn.Sequential(
            nn.Linear(
                encoder_width,
                correction_hidden_width,
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden_width,
                1,
            ),
        )

        nn.init.zeros_(
            self.extreme_gate.weight
        )

        prior_logit = math.log(
            extreme_prior
            / (1.0 - extreme_prior)
        )

        nn.init.constant_(
            self.extreme_gate.bias,
            prior_logit,
        )

        nn.init.zeros_(
            self.correction_head[-1].weight
        )

        nn.init.zeros_(
            self.correction_head[-1].bias
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        base_forecast: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        representation = (
            self.shock_encoder(
                input_tensor
            )
        )

        gate_logits = (
            self.extreme_gate(
                representation
            ).squeeze(-1)
        )

        gate_probability = torch.sigmoid(
            gate_logits
        )

        residual_correction = (
            self.correction_head(
                representation
            ).squeeze(-1)
        )

        final_forecast = (
            base_forecast
            + gate_probability
            * residual_correction
        )

        return {
            "gate_logits": gate_logits,
            "gate_probability": gate_probability,
            "residual_correction": residual_correction,
            "final_forecast": final_forecast,
        }


class UngatedCrossfeedAdapter(nn.Module):
    def __init__(
        self,
        input_dim: int = 141,
        encoder_width: int = 16,
        correction_hidden_width: int = 8,
    ) -> None:
        super().__init__()

        self.shock_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                encoder_width,
            ),
            nn.GELU(),
            nn.LayerNorm(
                encoder_width,
            ),
        )

        self.correction_head = nn.Sequential(
            nn.Linear(
                encoder_width,
                correction_hidden_width,
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden_width,
                1,
            ),
        )

        nn.init.zeros_(
            self.correction_head[-1].weight
        )

        nn.init.zeros_(
            self.correction_head[-1].bias
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        base_forecast: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        representation = (
            self.shock_encoder(
                input_tensor
            )
        )

        residual_correction = (
            self.correction_head(
                representation
            ).squeeze(-1)
        )

        final_forecast = (
            base_forecast
            + residual_correction
        )

        return {
            "residual_correction": residual_correction,
            "final_forecast": final_forecast,
        }

## Shared batch builder

In [9]:
alignment_weight_tensor = torch.from_numpy(
    selected_weight
).to(
    DEVICE
)

def build_inputs(
    indices: np.ndarray,
    *,
    training: bool,
    include_decoder: bool,
) -> dict[str, torch.Tensor]:
    if training:
        decoder_source = train_decoder_hidden
        recent_source = train_recent_features
        future_source = train_future_covariates
        base_source = train_base_forecast
        target_source = training_targets
        extreme_source = train_extreme_labels
    else:
        decoder_source = validation_decoder_hidden
        recent_source = validation_recent_features
        future_source = validation_future_covariates
        base_source = validation_base_forecast
        target_source = validation_targets
        extreme_source = validation_extreme_labels

    recent = torch.from_numpy(
        np.asarray(
            recent_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    future = torch.from_numpy(
        np.asarray(
            future_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    base = torch.from_numpy(
        np.asarray(
            base_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    target = torch.from_numpy(
        np.asarray(
            target_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    extreme = torch.from_numpy(
        np.asarray(
            extreme_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    recent_expanded = (
        recent
        .unsqueeze(1)
        .expand(
            -1,
            FORECAST_HORIZON,
            -1,
        )
    )

    if include_decoder:
        raw_decoder = torch.from_numpy(
            np.asarray(
                decoder_source[indices],
                dtype=np.float32,
            )
        ).to(DEVICE)

        decoder_features = torch.einsum(
            "bpk,hpk->bhk",
            raw_decoder[:, 0, :, :],
            alignment_weight_tensor,
        )

        input_tensor = torch.cat(
            [
                recent_expanded,
                decoder_features,
                future,
            ],
            dim=-1,
        )
    else:
        input_tensor = torch.cat(
            [
                recent_expanded,
                future,
            ],
            dim=-1,
        )

    return {
        "input": input_tensor,
        "base": base,
        "target": target,
        "extreme": extreme,
    }

## Shared validation evaluator

In [10]:
BATCH_SIZE = 512

hp_metrics = forecasting_metrics(
    validation_targets,
    np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    ),
)

hp_extreme_mse = masked_mse(
    validation_targets,
    np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    ),
    validation_extreme_mask,
)

hp_normal_mse = masked_mse(
    validation_targets,
    np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    ),
    validation_normal_mask,
)

def evaluate_model(
    model: nn.Module,
    *,
    include_decoder: bool,
    gated: bool,
) -> dict[str, Any]:
    model.eval()

    forecast = np.empty(
        (
            VALIDATION_SAMPLE_COUNT,
            FORECAST_HORIZON,
        ),
        dtype=np.float32,
    )

    gate_probability = (
        np.empty_like(
            forecast
        )
        if gated
        else None
    )

    residual_correction = np.empty_like(
        forecast
    )

    with torch.no_grad():
        for start in range(
            0,
            VALIDATION_SAMPLE_COUNT,
            BATCH_SIZE,
        ):
            end = min(
                start + BATCH_SIZE,
                VALIDATION_SAMPLE_COUNT,
            )

            indices = np.arange(
                start,
                end,
                dtype=np.int64,
            )

            batch = build_inputs(
                indices,
                training=False,
                include_decoder=include_decoder,
            )

            output = model(
                input_tensor=batch["input"],
                base_forecast=batch["base"],
            )

            forecast[start:end] = (
                output[
                    "final_forecast"
                ]
                .detach()
                .cpu()
                .numpy()
            )

            residual_correction[start:end] = (
                output[
                    "residual_correction"
                ]
                .detach()
                .cpu()
                .numpy()
            )

            if gated:
                gate_probability[start:end] = (
                    output[
                        "gate_probability"
                    ]
                    .detach()
                    .cpu()
                    .numpy()
                )

            del batch, output

    overall = forecasting_metrics(
        validation_targets,
        forecast,
    )

    extreme_mse = masked_mse(
        validation_targets,
        forecast,
        validation_extreme_mask,
    )

    normal_mse = masked_mse(
        validation_targets,
        forecast,
        validation_normal_mask,
    )

    overall_ratio = float(
        overall["mse"]
        / hp_metrics["mse"]
    )

    extreme_ratio = float(
        extreme_mse
        / hp_extreme_mse
    )

    balanced_score = float(
        0.5
        * (
            overall_ratio
            + extreme_ratio
        )
    )

    result = {
        "forecast": forecast,
        "residual_correction": residual_correction,
        "overall_metrics": overall,
        "extreme_mse": extreme_mse,
        "normal_mse": normal_mse,
        "balanced_score": balanced_score,
    }

    if gated:
        labels = np.asarray(
            validation_extreme_labels,
            dtype=np.uint8,
        ).ravel()

        scores = (
            gate_probability.ravel()
        )

        result.update(
            {
                "gate_probability": gate_probability,
                "gate_auroc": float(
                    roc_auc_score(
                        labels,
                        scores,
                    )
                ),
                "gate_auprc": float(
                    average_precision_score(
                        labels,
                        scores,
                    )
                ),
            }
        )

    return result

## Best-overall Phase 08 checkpoint audit

The Phase 08 best-overall checkpoint is evaluated only to document the trade-off created by the pre-registered balanced selection rule.

In [11]:
training_extreme_prior = float(
    phase07_summary[
        "training_extreme_rate"
    ]
)

best_overall_model = GatedAdapter(
    input_dim=141,
    extreme_prior=training_extreme_prior,
).to(DEVICE)

best_overall_checkpoint = torch.load(
    PHASE08_CHECKPOINT_DIR
    / "best_overall_state.pt",
    map_location=DEVICE,
    weights_only=False,
)

best_overall_model.load_state_dict(
    best_overall_checkpoint[
        "model_state_dict"
    ]
)

best_overall_result = evaluate_model(
    best_overall_model,
    include_decoder=True,
    gated=True,
)

del best_overall_model
torch.cuda.empty_cache()

## Ablation optimization contract

Both ablations inherit the Phase 08 optimization settings:

- batch size 512;
- AdamW;
- learning rate $10^{-3}$;
- weight decay $10^{-4}$;
- gradient clipping at 1.0;
- maximum 30 epochs;
- `ReduceLROnPlateau`;
- six-epoch early stopping on the same balanced validation score.

The retail-only gated model retains the normalized gate loss.  
The ungated cross-feed model optimizes normalized forecast MSE only.

In [12]:
INITIAL_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6
SCHEDULER_PATIENCE = 2
GRADIENT_CLIP_NORM = 1.0

training_base_mse = float(
    phase08_training_contract[
        "training_hp_reference_mse"
    ]
)

gate_prior_entropy = float(
    phase08_training_contract[
        "gate_prior_entropy"
    ]
)

forecast_loss_function = nn.MSELoss()
gate_loss_function = nn.BCEWithLogitsLoss()

def train_ablation(
    *,
    name: str,
    model: nn.Module,
    include_decoder: bool,
    gated: bool,
    seed_offset: int,
) -> dict[str, Any]:
    best_path = (
        CHECKPOINT_DIR
        / f"{name}_best_state.pt"
    )

    last_path = (
        CHECKPOINT_DIR
        / f"{name}_last_state.pt"
    )

    result_path = (
        RUN_DIR
        / f"{name}_result.json"
    )

    if result_path.exists() and best_path.exists():
        saved_result = json.loads(
            result_path.read_text(
                encoding="utf-8"
            )
        )

        best_checkpoint = torch.load(
            best_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(
            best_checkpoint[
                "model_state_dict"
            ]
        )

        final_evaluation = evaluate_model(
            model,
            include_decoder=include_decoder,
            gated=gated,
        )

        return {
            "history": saved_result["history"],
            "best_epoch": int(
                saved_result["best_epoch"]
            ),
            "training_runtime_seconds": float(
                saved_result[
                    "training_runtime_seconds"
                ]
            ),
            "evaluation": final_evaluation,
            "best_checkpoint": str(
                best_path
            ),
        }

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=INITIAL_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=SCHEDULER_PATIENCE,
            min_lr=1e-5,
        )
    )

    start_epoch = 0
    best_score = float("inf")
    best_epoch = None
    patience_counter = 0
    history: list[dict[str, Any]] = []

    if last_path.exists():
        checkpoint = torch.load(
            last_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        scheduler.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

        start_epoch = int(
            checkpoint["epoch"]
        )

        best_score = float(
            checkpoint[
                "best_score"
            ]
        )

        best_epoch = checkpoint[
            "best_epoch"
        ]

        patience_counter = int(
            checkpoint[
                "patience_counter"
            ]
        )

        history = list(
            checkpoint[
                "history"
            ]
        )

    num_batches = math.ceil(
        TRAINING_SAMPLE_COUNT
        / BATCH_SIZE
    )

    training_start_time = time.perf_counter()

    for epoch in range(
        start_epoch,
        MAX_EPOCHS,
    ):
        model.train()

        epoch_rng = np.random.default_rng(
            BASE_SEED
            + seed_offset
            + epoch
        )

        permutation = epoch_rng.permutation(
            TRAINING_SAMPLE_COUNT
        )

        epoch_forecast_loss_sum = 0.0
        epoch_gate_loss_sum = 0.0
        epoch_sample_count = 0

        for batch_index in range(
            num_batches
        ):
            batch_start = (
                batch_index
                * BATCH_SIZE
            )

            batch_end = min(
                batch_start
                + BATCH_SIZE,
                TRAINING_SAMPLE_COUNT,
            )

            indices = permutation[
                batch_start:batch_end
            ]

            batch = build_inputs(
                indices,
                training=True,
                include_decoder=include_decoder,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            output = model(
                input_tensor=batch["input"],
                base_forecast=batch["base"],
            )

            raw_forecast_loss = (
                forecast_loss_function(
                    output[
                        "final_forecast"
                    ],
                    batch["target"],
                )
            )

            normalized_forecast_loss = (
                raw_forecast_loss
                / training_base_mse
            )

            if gated:
                raw_gate_loss = (
                    gate_loss_function(
                        output[
                            "gate_logits"
                        ],
                        batch["extreme"],
                    )
                )

                total_loss = (
                    normalized_forecast_loss
                    + raw_gate_loss
                    / gate_prior_entropy
                )
            else:
                raw_gate_loss = None
                total_loss = (
                    normalized_forecast_loss
                )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP_NORM,
            )

            optimizer.step()

            current_batch_size = (
                batch_end
                - batch_start
            )

            epoch_forecast_loss_sum += (
                float(
                    raw_forecast_loss
                    .detach()
                    .item()
                )
                * current_batch_size
            )

            if gated:
                epoch_gate_loss_sum += (
                    float(
                        raw_gate_loss
                        .detach()
                        .item()
                    )
                    * current_batch_size
                )

            epoch_sample_count += (
                current_batch_size
            )

            del (
                batch,
                output,
                raw_forecast_loss,
                normalized_forecast_loss,
                raw_gate_loss,
                total_loss,
            )

        validation_result = (
            evaluate_model(
                model,
                include_decoder=include_decoder,
                gated=gated,
            )
        )

        current_score = float(
            validation_result[
                "balanced_score"
            ]
        )

        history.append(
            {
                "epoch": epoch + 1,
                "train_forecast_mse": (
                    epoch_forecast_loss_sum
                    / epoch_sample_count
                ),
                "train_gate_bce": (
                    epoch_gate_loss_sum
                    / epoch_sample_count
                    if gated
                    else None
                ),
                "validation_mse": (
                    validation_result[
                        "overall_metrics"
                    ]["mse"]
                ),
                "validation_extreme_mse": (
                    validation_result[
                        "extreme_mse"
                    ]
                ),
                "validation_normal_mse": (
                    validation_result[
                        "normal_mse"
                    ]
                ),
                "validation_balanced_score": (
                    current_score
                ),
                "learning_rate": float(
                    optimizer.param_groups[0][
                        "lr"
                    ]
                ),
            }
        )

        if (
            current_score
            < best_score
            - 1e-8
        ):
            best_score = current_score
            best_epoch = epoch + 1
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": best_epoch,
                    "best_score": best_score,
                },
                best_path,
            )
        else:
            patience_counter += 1

        scheduler.step(
            current_score
        )

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "epoch": epoch + 1,
                "best_score": best_score,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "history": history,
            },
            last_path,
        )

        print(
            f"{name} | "
            f"epoch {epoch + 1:02d} | "
            f"val MSE="
            f"{validation_result['overall_metrics']['mse']:.6f} | "
            f"extreme MSE="
            f"{validation_result['extreme_mse']:.6f} | "
            f"balanced="
            f"{current_score:.6f}"
        )

        if (
            patience_counter
            >= EARLY_STOPPING_PATIENCE
        ):
            break

    training_runtime_seconds = (
        time.perf_counter()
        - training_start_time
    )

    assert best_path.exists()

    best_checkpoint = torch.load(
        best_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )

    final_evaluation = evaluate_model(
        model,
        include_decoder=include_decoder,
        gated=gated,
    )

    saved_result = {
        "name": name,
        "best_epoch": int(
            best_checkpoint["epoch"]
        ),
        "training_runtime_seconds": float(
            training_runtime_seconds
        ),
        "history": history,
    }

    result_path.write_text(
        json.dumps(
            saved_result,
            indent=2,
        ),
        encoding="utf-8",
    )

    return {
        "history": history,
        "best_epoch": int(
            best_checkpoint["epoch"]
        ),
        "training_runtime_seconds": float(
            training_runtime_seconds
        ),
        "evaluation": final_evaluation,
        "best_checkpoint": str(
            best_path
        ),
    }

## Ablation A — Retail-only gated adapter

This model removes the 128-dimensional decoder cross-feed while keeping the recent-demand, future-known, gate, and correction mechanisms.

In [13]:
set_global_seed(
    BASE_SEED + 1000
)

retail_only_model = GatedAdapter(
    input_dim=13,
    extreme_prior=training_extreme_prior,
).to(DEVICE)

retail_only_parameter_count = sum(
    parameter.numel()
    for parameter in retail_only_model.parameters()
    if parameter.requires_grad
)

retail_only_result = train_ablation(
    name="retail_only_gated",
    model=retail_only_model,
    include_decoder=False,
    gated=True,
    seed_offset=1000,
)

del retail_only_model
torch.cuda.empty_cache()

retail_only_gated | epoch 01 | val MSE=4.377885 | extreme MSE=39.714502 | balanced=0.996108
retail_only_gated | epoch 02 | val MSE=4.377589 | extreme MSE=39.679709 | balanced=0.995639
retail_only_gated | epoch 03 | val MSE=4.376863 | extreme MSE=39.541379 | balanced=0.993827
retail_only_gated | epoch 04 | val MSE=4.378015 | extreme MSE=39.576342 | balanced=0.994395
retail_only_gated | epoch 05 | val MSE=4.377310 | extreme MSE=39.581459 | balanced=0.994379
retail_only_gated | epoch 06 | val MSE=4.376534 | extreme MSE=39.417918 | balanced=0.992246
retail_only_gated | epoch 07 | val MSE=4.375859 | extreme MSE=39.430987 | balanced=0.992332
retail_only_gated | epoch 08 | val MSE=4.374604 | extreme MSE=39.478459 | balanced=0.992782
retail_only_gated | epoch 09 | val MSE=4.373474 | extreme MSE=39.512838 | balanced=0.993083
retail_only_gated | epoch 10 | val MSE=4.373026 | extreme MSE=39.424368 | balanced=0.991926
retail_only_gated | epoch 11 | val MSE=4.372914 | extreme MSE=39.441791 | balanc

## Ablation B — Ungated cross-feed residual adapter

This model retains the full cross-feed input but removes the gate branch and explicit extreme-classification objective.

In [14]:
set_global_seed(
    BASE_SEED + 2000
)

ungated_crossfeed_model = (
    UngatedCrossfeedAdapter(
        input_dim=141,
    ).to(DEVICE)
)

ungated_crossfeed_parameter_count = sum(
    parameter.numel()
    for parameter in ungated_crossfeed_model.parameters()
    if parameter.requires_grad
)

ungated_crossfeed_result = train_ablation(
    name="ungated_crossfeed",
    model=ungated_crossfeed_model,
    include_decoder=True,
    gated=False,
    seed_offset=2000,
)

del ungated_crossfeed_model
torch.cuda.empty_cache()

ungated_crossfeed | epoch 01 | val MSE=4.361047 | extreme MSE=39.486870 | balanced=0.991340
ungated_crossfeed | epoch 02 | val MSE=4.351729 | extreme MSE=39.325411 | balanced=0.988259
ungated_crossfeed | epoch 03 | val MSE=4.350461 | extreme MSE=39.483474 | balanced=0.990090
ungated_crossfeed | epoch 04 | val MSE=4.338330 | extreme MSE=38.866422 | balanced=0.980991
ungated_crossfeed | epoch 05 | val MSE=4.346815 | extreme MSE=38.914054 | balanced=0.982555
ungated_crossfeed | epoch 06 | val MSE=4.355833 | extreme MSE=38.245151 | balanced=0.975222
ungated_crossfeed | epoch 07 | val MSE=4.343145 | extreme MSE=39.211403 | balanced=0.985854
ungated_crossfeed | epoch 08 | val MSE=4.333191 | extreme MSE=39.059970 | balanced=0.982825
ungated_crossfeed | epoch 09 | val MSE=4.333890 | extreme MSE=38.885088 | balanced=0.980718
ungated_crossfeed | epoch 10 | val MSE=4.315652 | extreme MSE=38.923958 | balanced=0.979123
ungated_crossfeed | epoch 11 | val MSE=4.318075 | extreme MSE=38.984839 | balanc

## Comparative validation table

In [15]:
primary_ea_metrics = (
    phase08_summary[
        "ea_ttm_validation_metrics"
    ]
)

primary_ea_extreme_mse = float(
    phase08_summary[
        "ea_ttm_validation_extreme_mse"
    ]
)

primary_ea_normal_mse = float(
    phase08_summary[
        "ea_ttm_validation_normal_mse"
    ]
)

primary_ea_balanced = float(
    phase08_summary[
        "ea_ttm_balanced_score"
    ]
)

retail_eval = (
    retail_only_result[
        "evaluation"
    ]
)

ungated_eval = (
    ungated_crossfeed_result[
        "evaluation"
    ]
)

model_comparison_records = [
    {
        "model": "Frozen HP-TTM",
        "checkpoint_role": "baseline",
        "mse": hp_metrics["mse"],
        "rmse": hp_metrics["rmse"],
        "mae": hp_metrics["mae"],
        "wape": hp_metrics["wape"],
        "extreme_mse": hp_extreme_mse,
        "normal_mse": hp_normal_mse,
        "balanced_score": 1.0,
    },
    {
        "model": "EA-TTM",
        "checkpoint_role": "primary_balanced",
        "mse": primary_ea_metrics["mse"],
        "rmse": primary_ea_metrics["rmse"],
        "mae": primary_ea_metrics["mae"],
        "wape": primary_ea_metrics["wape"],
        "extreme_mse": primary_ea_extreme_mse,
        "normal_mse": primary_ea_normal_mse,
        "balanced_score": primary_ea_balanced,
    },
    {
        "model": "EA-TTM",
        "checkpoint_role": "best_overall_diagnostic",
        "mse": best_overall_result["overall_metrics"]["mse"],
        "rmse": best_overall_result["overall_metrics"]["rmse"],
        "mae": best_overall_result["overall_metrics"]["mae"],
        "wape": best_overall_result["overall_metrics"]["wape"],
        "extreme_mse": best_overall_result["extreme_mse"],
        "normal_mse": best_overall_result["normal_mse"],
        "balanced_score": best_overall_result["balanced_score"],
    },
    {
        "model": "Retail-only gated adapter",
        "checkpoint_role": "ablation",
        "mse": retail_eval["overall_metrics"]["mse"],
        "rmse": retail_eval["overall_metrics"]["rmse"],
        "mae": retail_eval["overall_metrics"]["mae"],
        "wape": retail_eval["overall_metrics"]["wape"],
        "extreme_mse": retail_eval["extreme_mse"],
        "normal_mse": retail_eval["normal_mse"],
        "balanced_score": retail_eval["balanced_score"],
    },
    {
        "model": "Ungated cross-feed adapter",
        "checkpoint_role": "ablation",
        "mse": ungated_eval["overall_metrics"]["mse"],
        "rmse": ungated_eval["overall_metrics"]["rmse"],
        "mae": ungated_eval["overall_metrics"]["mae"],
        "wape": ungated_eval["overall_metrics"]["wape"],
        "extreme_mse": ungated_eval["extreme_mse"],
        "normal_mse": ungated_eval["normal_mse"],
        "balanced_score": ungated_eval["balanced_score"],
    },
]

model_comparison_table = pd.DataFrame(
    model_comparison_records
)

model_comparison_table

,model,checkpoint_role,mse,rmse,mae,wape,extreme_mse,normal_mse,balanced_score
0,Frozen HP-TTM,baseline,4.381208,2.093133,1.007348,0.726575,39.995519,2.216475,1.000000
1,EA-TTM,primary_balanced,4.368124,2.090006,1.022395,0.737428,39.210961,2.250284,0.988699
2,EA-TTM,best_overall_diagnostic,4.366784,2.089685,1.019450,0.735304,39.296558,2.243660,0.989616
3,Retail-only gated adapter,ablation,4.371620,2.090842,1.021369,0.736688,39.291771,2.249081,0.990108
4,Ungated cross-feed adapter,ablation,4.355833,2.087063,1.025313,0.739533,38.245151,2.295950,0.975222


## Incremental contribution evidence

In [16]:
decoder_crossfeed_increment = {
    "balanced_score_change_vs_retail_only_percent": (
        improvement_percent(
            retail_eval[
                "balanced_score"
            ],
            primary_ea_balanced,
        )
    ),
    "overall_mse_change_vs_retail_only_percent": (
        improvement_percent(
            retail_eval[
                "overall_metrics"
            ]["mse"],
            primary_ea_metrics["mse"],
        )
    ),
    "extreme_mse_change_vs_retail_only_percent": (
        improvement_percent(
            retail_eval[
                "extreme_mse"
            ],
            primary_ea_extreme_mse,
        )
    ),
}

gate_increment = {
    "balanced_score_change_vs_ungated_percent": (
        improvement_percent(
            ungated_eval[
                "balanced_score"
            ],
            primary_ea_balanced,
        )
    ),
    "overall_mse_change_vs_ungated_percent": (
        improvement_percent(
            ungated_eval[
                "overall_metrics"
            ]["mse"],
            primary_ea_metrics["mse"],
        )
    ),
    "extreme_mse_change_vs_ungated_percent": (
        improvement_percent(
            ungated_eval[
                "extreme_mse"
            ],
            primary_ea_extreme_mse,
        )
    ),
}

mechanism_evidence = {
    "extreme_prevalence": (
        primary_extreme_prevalence
    ),
    "gate_auroc": primary_gate_auroc,
    "gate_auprc": primary_gate_auprc,
    "gate_auprc_lift_vs_prevalence": (
        primary_auprc_lift
    ),
    "mean_abs_effective_correction_extreme": (
        mean_abs_effective_correction_extreme
    ),
    "mean_abs_effective_correction_normal": (
        mean_abs_effective_correction_normal
    ),
    "effective_correction_extreme_to_normal_ratio": (
        effective_correction_ratio_extreme_to_normal
    ),
    "extreme_point_squared_error_improvement_rate": (
        extreme_point_improvement_rate
    ),
    "normal_point_squared_error_improvement_rate": (
        normal_point_improvement_rate
    ),
    "overall_correction_direction_agreement": (
        correction_direction_agreement
    ),
    "extreme_correction_direction_agreement": (
        extreme_correction_direction_agreement
    ),
}

## Freeze development state

No Phase 09 result is permitted to change the primary Phase 08 model. The lock records that the next model evaluation is the single final test.

In [17]:
MODEL_DEVELOPMENT_LOCK_PATH = (
    METADATA_DIR
    / "09_MODEL_DEVELOPMENT_LOCK.json"
)

ABLATION_COMPARISON_PATH = (
    RESULTS_TABLE_DIR
    / "09_VALIDATION_ABLATION_COMPARISON.csv"
)

MECHANISM_EVIDENCE_PATH = (
    RESULTS_TABLE_DIR
    / "09_MECHANISM_EVIDENCE.csv"
)

INCREMENTAL_CONTRIBUTION_PATH = (
    RESULTS_TABLE_DIR
    / "09_INCREMENTAL_CONTRIBUTION.csv"
)

RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "09_RUN_SUMMARY.json"
)

model_comparison_table.to_csv(
    ABLATION_COMPARISON_PATH,
    index=False,
)

pd.DataFrame(
    [
        {
            "metric": key,
            "value": value,
        }
        for key, value in (
            mechanism_evidence.items()
        )
    ]
).to_csv(
    MECHANISM_EVIDENCE_PATH,
    index=False,
)

incremental_records = []

for component, evidence in (
    (
        "decoder_crossfeed",
        decoder_crossfeed_increment,
    ),
    (
        "extreme_gate",
        gate_increment,
    ),
):
    for metric, value in evidence.items():
        incremental_records.append(
            {
                "component": component,
                "metric": metric,
                "value_percent": value,
            }
        )

pd.DataFrame(
    incremental_records
).to_csv(
    INCREMENTAL_CONTRIBUTION_PATH,
    index=False,
)

development_lock = {
    "status": "MODEL_DEVELOPMENT_LOCKED",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "phase08_training_config_sha256": TRAINING_CONFIG_SHA256,
    "primary_model": (
        "Phase 08 best-balanced EA-TTM"
    ),
    "primary_epoch": PRIMARY_EPOCH,
    "primary_adapter_state": str(
        PHASE08_MODEL_DIR
        / "adapter_state.pt"
    ),
    "selection_rule": (
        "balanced overall/extreme validation MSE ratio"
    ),
    "phase09_ablation_results_may_change_primary_model": False,
    "post_lock_tuning_permitted": False,
    "locked_final_test_interval": [
        "d_1914",
        "d_1941",
    ],
    "next_evaluation": (
        "single locked final-test evaluation"
    ),
}

development_lock_payload = json.dumps(
    development_lock,
    sort_keys=True,
    separators=(",", ":"),
).encode(
    "utf-8"
)

development_lock_sha256 = hashlib.sha256(
    development_lock_payload
).hexdigest()

development_lock[
    "development_lock_sha256"
] = development_lock_sha256

MODEL_DEVELOPMENT_LOCK_PATH.write_text(
    json.dumps(
        development_lock,
        indent=2,
    ),
    encoding="utf-8",
)

894

In [18]:
run_summary = {
    "status": "PHASE_09_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "development_lock_sha256": development_lock_sha256,
    "primary_model": "Phase 08 best-balanced EA-TTM",
    "primary_epoch": PRIMARY_EPOCH,
    "best_overall_epoch": BEST_OVERALL_EPOCH,
    "best_extreme_epoch": BEST_EXTREME_EPOCH,
    "primary_validation": {
        "mse": primary_ea_metrics["mse"],
        "extreme_mse": primary_ea_extreme_mse,
        "normal_mse": primary_ea_normal_mse,
        "balanced_score": primary_ea_balanced,
    },
    "best_overall_checkpoint_validation": {
        "mse": best_overall_result["overall_metrics"]["mse"],
        "extreme_mse": best_overall_result["extreme_mse"],
        "normal_mse": best_overall_result["normal_mse"],
        "balanced_score": best_overall_result["balanced_score"],
    },
    "retail_only_validation": {
        "best_epoch": retail_only_result["best_epoch"],
        "parameter_count": retail_only_parameter_count,
        "mse": retail_eval["overall_metrics"]["mse"],
        "extreme_mse": retail_eval["extreme_mse"],
        "normal_mse": retail_eval["normal_mse"],
        "balanced_score": retail_eval["balanced_score"],
        "gate_auroc": retail_eval["gate_auroc"],
        "gate_auprc": retail_eval["gate_auprc"],
    },
    "ungated_crossfeed_validation": {
        "best_epoch": ungated_crossfeed_result["best_epoch"],
        "parameter_count": ungated_crossfeed_parameter_count,
        "mse": ungated_eval["overall_metrics"]["mse"],
        "extreme_mse": ungated_eval["extreme_mse"],
        "normal_mse": ungated_eval["normal_mse"],
        "balanced_score": ungated_eval["balanced_score"],
    },
    "mechanism_evidence": mechanism_evidence,
    "decoder_crossfeed_increment_percent": decoder_crossfeed_increment,
    "gate_increment_percent": gate_increment,
    "locked_final_test_used": False,
    "post_phase09_tuning_permitted": False,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(
        run_summary,
        indent=2,
    ),
    encoding="utf-8",
)

2616

In [19]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    (
        "Phase 09 — Validation ablations and "
        "mechanism audit completed."
    ),
    "",
    "## Primary model lock",
    "- Primary model: Phase 08 best-balanced EA-TTM",
    f"- Primary epoch: {PRIMARY_EPOCH}",
    "- Phase 09 diagnostics cannot change the primary model",
    "- Post-lock tuning: prohibited",
    "",
    "## Mechanism evidence",
    f"- Gate AUROC: {primary_gate_auroc:.6f}",
    f"- Gate AUPRC: {primary_gate_auprc:.6f}",
    f"- AUPRC lift over prevalence: {primary_auprc_lift:.3f}x",
    (
        "- Effective correction extreme/normal ratio: "
        f"{effective_correction_ratio_extreme_to_normal:.6f}"
    ),
    "",
    "## Ablations",
    (
        "- Retail-only gated balanced score: "
        f"{retail_eval['balanced_score']:.6f}"
    ),
    (
        "- Ungated cross-feed balanced score: "
        f"{ungated_eval['balanced_score']:.6f}"
    ),
    (
        "- Full EA-TTM balanced score: "
        f"{primary_ea_balanced:.6f}"
    ),
    "",
    "## Next phase",
    (
        "Single locked final-test evaluation on "
        "d_1914 through d_1941."
    ),
]

status_path.write_text(
    "\n".join(
        status_lines
    ),
    encoding="utf-8",
)

661

## Phase validation

Phase 09 passes only if:

$$
\begin{aligned}
&\text{Phase 08 primary checkpoint remains unchanged}
\\
&\land\
\text{retail-only ablation is trained independently}
\\
&\land\
\text{ungated cross-feed ablation is trained independently}
\\
&\land\
\text{mechanism evidence is persisted}
\\
&\land\
\text{model-development lock is persisted}
\\
&\land\
\text{locked final-test demand remains unused}.
\end{aligned}
$$

In [20]:
phase_validation = {
    "primary_model_unchanged": (
        development_lock[
            "primary_model"
        ]
        == "Phase 08 best-balanced EA-TTM"
    ),
    "retail_only_checkpoint_exists": Path(
        retail_only_result[
            "best_checkpoint"
        ]
    ).exists(),
    "ungated_checkpoint_exists": Path(
        ungated_crossfeed_result[
            "best_checkpoint"
        ]
    ).exists(),
    "mechanism_evidence_saved": (
        MECHANISM_EVIDENCE_PATH.exists()
    ),
    "ablation_comparison_saved": (
        ABLATION_COMPARISON_PATH.exists()
    ),
    "development_lock_saved": (
        MODEL_DEVELOPMENT_LOCK_PATH.exists()
    ),
    "post_lock_tuning_disabled": (
        development_lock[
            "post_lock_tuning_permitted"
        ] is False
    ),
    "locked_test_unused": True,
    "run_summary_saved": (
        RUN_SUMMARY_PATH.exists()
    ),
}

assert all(
    phase_validation.values()
), (
    f"Phase 09 validation failed: "
    f"{phase_validation}"
)

phase_validation

{'primary_model_unchanged': True,
 'retail_only_checkpoint_exists': True,
 'ungated_checkpoint_exists': True,
 'mechanism_evidence_saved': True,
 'ablation_comparison_saved': True,
 'development_lock_saved': True,
 'post_lock_tuning_disabled': True,
 'locked_test_unused': True,
 'run_summary_saved': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [21]:
return_block = {
    "status": "PHASE_09_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "development_lock_sha256": (
        development_lock_sha256
    ),
    "primary_model_lock": {
        "model": (
            "Phase 08 best-balanced EA-TTM"
        ),
        "epoch": PRIMARY_EPOCH,
        "post_lock_tuning_permitted": False,
    },
    "checkpoint_comparison": {
        "primary_balanced": {
            "epoch": PRIMARY_EPOCH,
            "mse": primary_ea_metrics["mse"],
            "extreme_mse": primary_ea_extreme_mse,
            "normal_mse": primary_ea_normal_mse,
            "balanced_score": primary_ea_balanced,
        },
        "best_overall_diagnostic": {
            "epoch": BEST_OVERALL_EPOCH,
            "mse": best_overall_result[
                "overall_metrics"
            ]["mse"],
            "extreme_mse": best_overall_result[
                "extreme_mse"
            ],
            "normal_mse": best_overall_result[
                "normal_mse"
            ],
            "balanced_score": best_overall_result[
                "balanced_score"
            ],
        },
    },
    "ablation_results": {
        "retail_only_gated": {
            "best_epoch": retail_only_result[
                "best_epoch"
            ],
            "trainable_parameters": (
                retail_only_parameter_count
            ),
            "mse": retail_eval[
                "overall_metrics"
            ]["mse"],
            "extreme_mse": retail_eval[
                "extreme_mse"
            ],
            "normal_mse": retail_eval[
                "normal_mse"
            ],
            "balanced_score": retail_eval[
                "balanced_score"
            ],
            "gate_auroc": retail_eval[
                "gate_auroc"
            ],
            "gate_auprc": retail_eval[
                "gate_auprc"
            ],
        },
        "ungated_crossfeed": {
            "best_epoch": (
                ungated_crossfeed_result[
                    "best_epoch"
                ]
            ),
            "trainable_parameters": (
                ungated_crossfeed_parameter_count
            ),
            "mse": ungated_eval[
                "overall_metrics"
            ]["mse"],
            "extreme_mse": ungated_eval[
                "extreme_mse"
            ],
            "normal_mse": ungated_eval[
                "normal_mse"
            ],
            "balanced_score": ungated_eval[
                "balanced_score"
            ],
        },
    },
    "mechanism_evidence": (
        mechanism_evidence
    ),
    "incremental_contribution_percent": {
        "decoder_crossfeed": (
            decoder_crossfeed_increment
        ),
        "extreme_gate": (
            gate_increment
        ),
    },
    "locked_final_test_used": False,
    "artifacts": {
        "model_development_lock": str(
            MODEL_DEVELOPMENT_LOCK_PATH
        ),
        "ablation_comparison": str(
            ABLATION_COMPARISON_PATH
        ),
        "mechanism_evidence": str(
            MECHANISM_EVIDENCE_PATH
        ),
        "incremental_contribution": str(
            INCREMENTAL_CONTRIBUTION_PATH
        ),
        "retail_only_best_checkpoint": (
            retail_only_result[
                "best_checkpoint"
            ]
        ),
        "ungated_crossfeed_best_checkpoint": (
            ungated_crossfeed_result[
                "best_checkpoint"
            ]
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 09 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 09 RETURN BLOCK
{
  "status": "PHASE_09_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "development_lock_sha256": "b68b706f78f96c9cf497369d009323c2bb2c894433c3edb0170aaf028d31a180",
  "primary_model_lock": {
    "model": "Phase 08 best-balanced EA-TTM",
    "epoch": 15,
    "post_lock_tuning_permitted": false
  },
  "checkpoint_comparison": {
    "primary_balanced": {
      "epoch": 15,
      "mse": 4.368124020733498,
      "extreme_mse": 39.21096076831241,
      "normal_mse": 2.2502839954626053,
      "balanced_score": 0.9886987815906065
    },
    "best_overall_diagnostic": {
      "epoch": 20,
      "mse": 4.366784101617184,
      "extreme_mse": 39.296557666782576,
      "normal_mse": 2.2436598260055907,
      "balanced_score": 0.9896159460532592
    }
  },
  "ablation_results": {
    "retail_only_gated": {
      "best_epoch": 17,
      "t